# BlackJAX real-data pixelized source demo

This notebook loads a real FITS dataset, then fits a lens mass model using a
pixelized Voronoi source (linear inversion inside the likelihood).

Topology (Delaunay) is cached once on the host via SciPy. A simple cache
refresh loop is provided to rebuild the cache if the lens parameters drift.


In [ ]:
# Keep JAX on CPU to avoid CUDA plugin warnings in minimal environments
import os
os.environ.setdefault("JAX_PLATFORM_NAME", "cpu")

import sys
sys.path.insert(0, "..")

import jax
import jax.numpy as jnp
import jax.random as jr
import jax.nn as jnn
import numpy as np
import matplotlib.pyplot as plt

import blackjax
from blackjax.adaptation.window_adaptation import window_adaptation

from jax_lens.lens.tracer import (
    TracerConfig,
    PlaneConfig,
    traced_grid_list,
)
from jax_lens.pipeline import create_likelihood_fn
from jax_lens.data import load_imaging_dataset
from jax_lens.pixelization import (
    PixelizationConfig,
    build_pixelization_cache_from_tracer,
    pixelized_source_reconstruction,
)

jax.config.update("jax_enable_x64", True)
plt.rcParams["figure.figsize"] = (12, 4)


In [ ]:
# Settings: quick run vs fuller run

quick_run = True

if quick_run:
    crop_size = 121
    n_side = 22  # ~484 seeds
    num_stages = 1
    num_adapt = 200
    num_samples = 200
    burn_in = 50
    refresh_threshold = 0.25
else:
    crop_size = 161
    n_side = 28  # ~784 seeds
    num_stages = 2
    num_adapt = 400
    num_samples = 400
    burn_in = 100
    refresh_threshold = 0.15


In [ ]:
# Utilities: grid cropping + parameter transforms

def create_grid(n_pixels: int, pixel_scale: float) -> jnp.ndarray:
    coords = jnp.linspace(-(n_pixels - 1) / 2 * pixel_scale, (n_pixels - 1) / 2 * pixel_scale, n_pixels)
    yy, xx = jnp.meshgrid(coords, coords, indexing="ij")
    return jnp.stack([yy, xx], axis=-1)

def crop_center(arr: jnp.ndarray, size: int) -> jnp.ndarray:
    if arr is None:
        return None
    ny, nx = arr.shape
    y0 = (ny - size) // 2
    x0 = (nx - size) // 2
    return arr[y0 : y0 + size, x0 : x0 + size]

def wrap_angle(phi: jnp.ndarray) -> jnp.ndarray:
    return (phi + jnp.pi) % (2 * jnp.pi) - jnp.pi

def softplus_pos(raw: jnp.ndarray, floor: float = 1e-3) -> jnp.ndarray:
    return floor + jnn.softplus(raw)

def inv_softplus_pos(val: jnp.ndarray, floor: float = 1e-3) -> jnp.ndarray:
    v = jnp.maximum(val - floor, 1e-8)
    return jnp.log(jnp.expm1(v))

def bounded_from_raw(raw: jnp.ndarray, low: float, high: float) -> jnp.ndarray:
    return low + (high - low) * jnn.sigmoid(raw)

def inv_bounded_from_raw(val: jnp.ndarray, low: float, high: float) -> jnp.ndarray:
    v = (val - low) / (high - low)
    v = jnp.clip(v, 1e-6, 1.0 - 1e-6)
    return jnp.log(v) - jnp.log1p(-v)


In [ ]:
# Load real dataset

data_dir = "/home/nataliehogg/Documents/Projects/cowls/M25/COSJ095921+020638/F150W"

ds = load_imaging_dataset(
    data_dir,
    data_filename="data.fits",
    noise_filename="noise_map.fits",
    psf_filename="psf.fits",
    mask_filename="mask_extra_galaxies.fits",
)

# Optionally crop to a smaller field for speed
if crop_size is not None:
    data = crop_center(ds.data, crop_size)
    noise_map = crop_center(ds.noise_map, crop_size)
    mask = crop_center(ds.mask, crop_size)
    grid = create_grid(crop_size, ds.pixel_scale)
else:
    data = ds.data
    noise_map = ds.noise_map
    mask = ds.mask
    grid = ds.grid

grid_flat = grid.reshape(-1, 2)
image_shape = data.shape

data_flat = data.reshape(-1)
noise_flat = noise_map.reshape(-1)

print("Image shape:", data.shape, "pixel_scale:", ds.pixel_scale)
print("PSF shape:", ds.psf.shape if ds.psf is not None else None)


In [ ]:
# Configure lens + pixelization

lens_plane = PlaneConfig(
    redshift=0.4694,
    light_profile_types=(),
    mass_profile_types=("sie",),
)
source_plane = PlaneConfig(
    redshift=1.0,
    light_profile_types=(),
    mass_profile_types=(),
)
config = TracerConfig(planes=(lens_plane, source_plane))

# Initial lens guess for cache construction
init_mass = {
    "centre": jnp.array([0.0, 0.0]),
    "einstein_radius": 1.0,
    "axis_ratio": 0.8,
    "angle": 0.4,
}
params_cache = {
    "planes": [
        {"mass": [init_mass], "light": []},
        {"mass": [], "light": []},
    ]
}

# Build seeds in the traced source-plane bounding box
traced_init = traced_grid_list(grid_flat, config, params_cache)[-1]
t = np.array(traced_init)
ymin, ymax = t[:, 0].min(), t[:, 0].max()
xmin, xmax = t[:, 1].min(), t[:, 1].max()

margin = 0.05
ys = np.linspace(ymin + margin, ymax - margin, n_side)
xs = np.linspace(xmin + margin, xmax - margin, n_side)
yy, xx = np.meshgrid(ys, xs, indexing="ij")
seeds = np.stack([yy.ravel(), xx.ravel()], axis=-1)

cache = build_pixelization_cache_from_tracer(
    grid=grid_flat,
    config=config,
    params=params_cache,
    seeds=seeds,
)

pix_cfg = PixelizationConfig(
    regularization_weight=1.0,
    solver="dense",
    seeds=jnp.asarray(seeds),
)


In [ ]:
# Sanity-check: traced grid coverage vs seeds

plt.figure(figsize=(6, 5))
plt.scatter(t[:, 1], t[:, 0], s=1, alpha=0.4, label="Traced points")
plt.scatter(seeds[:, 1], seeds[:, 0], s=8, alpha=0.6, label="Seeds")
plt.gca().invert_yaxis()
plt.legend()
plt.title("Source-plane coverage (traced points vs seeds)")
plt.show()


In [ ]:
# Build likelihood for pixelized source model

log_like_fn = create_likelihood_fn(
    config=config,
    grid=grid_flat,
    data=data_flat,
    noise_map=noise_flat,
    mask=mask,
    psf=ds.psf,
    image_shape=image_shape,
    source_mode="pixelization",
    pixelization=pix_cfg,
)


In [ ]:
# Parameterization: fit only lens mass parameters

def unpack_theta(theta: jnp.ndarray) -> dict:
    centre = theta[:2]
    einstein = softplus_pos(theta[2], floor=1e-3)
    axis_ratio = bounded_from_raw(theta[3], 0.2, 0.99)
    angle = wrap_angle(theta[4])

    return {
        "planes": [
            {
                "mass": [
                    {
                        "centre": centre,
                        "einstein_radius": einstein,
                        "axis_ratio": axis_ratio,
                        "angle": angle,
                    }
                ],
                "light": [],
            },
            {"mass": [], "light": []},
        ]
    }

theta_init = jnp.array([
    init_mass["centre"][0],
    init_mass["centre"][1],
    inv_softplus_pos(init_mass["einstein_radius"], floor=1e-3),
    inv_bounded_from_raw(init_mass["axis_ratio"], 0.2, 0.99),
    init_mass["angle"],
])

prior_loc = theta_init
prior_scale = jnp.array([0.2, 0.2, 0.4, 0.6, 0.6])

@jax.jit
def log_prior(theta: jnp.ndarray) -> jnp.ndarray:
    z = (theta - prior_loc) / prior_scale
    return -0.5 * jnp.sum(z**2) - jnp.sum(jnp.log(prior_scale * jnp.sqrt(2 * jnp.pi)))

def theta_drift(a: jnp.ndarray, b: jnp.ndarray) -> float:
    return float(jnp.linalg.norm(a - b))


In [ ]:
# Sampling with optional cache refresh

def sample_stage(rng_key, theta_start, cache):
    def log_posterior(theta: jnp.ndarray) -> jnp.ndarray:
        params = unpack_theta(theta)
        return log_like_fn(params, cache) + log_prior(theta)

    log_posterior_jit = jax.jit(log_posterior)

    rng_key, rng_adapt, rng_sample = jr.split(rng_key, 3)
    adapt = window_adaptation(
        blackjax.nuts,
        log_posterior_jit,
        initial_step_size=0.02,
        target_acceptance_rate=0.8,
        is_mass_matrix_diagonal=True,
    )
    adapt_results, adapt_info = adapt.run(rng_adapt, theta_start, num_steps=num_adapt)

    nuts = blackjax.nuts.differentiable(
        log_posterior_jit,
        **adapt_results.parameters,
    )

    @jax.jit
    def one_step(state, key):
        new_state, info = nuts.step(key, state)
        return new_state, (new_state, info)

    keys = jr.split(rng_sample, num_samples)
    state, (states, infos) = jax.lax.scan(one_step, adapt_results.state, keys)
    samples = states.position

    return samples, state, rng_key

def maybe_refresh_cache(cache, theta_ref, theta_new):
    drift = theta_drift(theta_ref, theta_new)
    if drift < refresh_threshold:
        return cache, theta_ref

    params_cache = unpack_theta(theta_new)
    cache = build_pixelization_cache_from_tracer(
        grid=grid_flat,
        config=config,
        params=params_cache,
        seeds=seeds,
)
    return cache, theta_new

rng = jr.PRNGKey(10)
theta_current = theta_init
cache_current = cache
theta_ref = theta_init

all_samples = []
for stage in range(num_stages):
    samples, state, rng = sample_stage(rng, theta_current, cache_current)
    all_samples.append(samples)

    mean_theta = jnp.mean(samples[burn_in:], axis=0)
    cache_current, theta_ref = maybe_refresh_cache(cache_current, theta_ref, mean_theta)

    theta_current = samples[-1]

samples = jnp.concatenate(all_samples, axis=0)


In [ ]:
# Posterior summary and reconstruction

posterior_tree = jax.vmap(unpack_theta)(samples[burn_in:])
mean_params = jax.tree.map(lambda x: jnp.mean(x, axis=0), posterior_tree)

print("Einstein radius (mean):", float(mean_params["planes"][0]["mass"][0]["einstein_radius"]))
print("Axis ratio (mean):", float(mean_params["planes"][0]["mass"][0]["axis_ratio"]))
print("Centre (mean y, x):", np.round(np.array(mean_params["planes"][0]["mass"][0]["centre"]), 3))

traced_mean = traced_grid_list(grid_flat, config, mean_params)[-1]
source_model, source_values = pixelized_source_reconstruction(
    points=traced_mean,
    seeds=jnp.asarray(seeds),
    cache=cache_current,
    data=data_flat,
    noise_map=noise_flat,
    regularization_weight=pix_cfg.regularization_weight,
    solver=pix_cfg.solver,
    psf=ds.psf,
    image_shape=image_shape,
    mask=mask,
)

model_recon = source_model.reshape(image_shape)

fig, axes = plt.subplots(2, 2, figsize=(10, 9))
axes = axes.ravel()

im0 = axes[0].imshow(data, origin="lower", cmap="magma")
axes[0].set_title("Data")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(model_recon, origin="lower", cmap="magma")
axes[1].set_title("Pixelized Reconstruction")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(data - np.array(model_recon), origin="lower", cmap="coolwarm")
axes[2].set_title("Residuals")
plt.colorbar(im2, ax=axes[2], fraction=0.046)

if mask is not None:
    im3 = axes[3].imshow(mask, origin="lower", cmap="gray")
    axes[3].set_title("Mask")

plt.tight_layout()
plt.show()

plt.figure(figsize=(5, 4))
plt.scatter(seeds[:, 1], seeds[:, 0], c=np.array(source_values), s=12, cmap="magma")
plt.gca().invert_yaxis()
plt.title("Recovered Source Values on Voronoi Seeds")
plt.colorbar()
plt.show()
